<a href="https://colab.research.google.com/github/Michael2004-ukpeh/ml-playground/blob/master/Pytorch_Fashion_MNIST_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [27]:
# Donwload training data from open datasets
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        # v2.Normalize(mean=(0.5,), std=(0.5,))
    ])
)
# Download test data from open datasets
test_data =  datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True)
        ])
)
test_data

Dataset FashionMNIST
    Number of datapoints: 10000
    Root location: data
    Split: Test
    StandardTransform
Transform: Compose(
                 ToImage()
                 ToDtype(scale=True)
           )

# Load data into 64 batches


In [28]:
batch_size = 64

# Create data loaders
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
  print(f"Shape of X [N, C, H, W]: {X.shape}")
  print(f"Shape of y: {y.shape} {y.dtype}")
  break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


# Create Model

In [29]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [30]:
# Define model
class NeuralNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = nn.Flatten()
    self.linear_relu_stack = nn.Sequential(
        nn.Linear(28*28, 512),
        nn.ReLU(),
        nn.Linear(512, 512),
        nn.ReLU(),
        nn.Linear(512, 10)
    )

  def forward(self, x):
      x = self.flatten(x)
      logits = self.linear_relu_stack(x)
      return logits

model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


# Loss function and Optimizer

In [31]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

# Training Loop

In [32]:
def train(dataloader,model,loss_fn,optimizer):
  size = len(dataloader.dataset)
  model.train()
  for batch, (X, y) in enumerate(dataloader):
    X, y = X.to(device), y.to(device)

    # Compute loss
    pred = model(X)
    loss = loss_fn(pred, y)

    # Backpropagation
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if batch % 100 == 0:
      loss, current = loss.item(), (batch + 1) * len(X)
      print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")



# Test loop

In [33]:
def test(dataloader,model, loss_fn):
         size = len(dataloader.dataset)
         num_batches = len(dataloader)
         model.eval()
         test_loss, correct = 0, 0
         with torch.no_grad():
          for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
         test_loss /= num_batches
         correct /= size
         print(f"Test Error: \n Accuracy: {(100 * correct) :> 0.1f}%, Avg loss: {test_loss:>8f} \n")


# Training

In [34]:
epochs = 5
for t in range(epochs):
  print(f"Epoch {t + 1}\n--------------")
  train(train_dataloader, model, loss_fn, optimizer)
  test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
--------------
loss: 2.292929 [   64/60000]
loss: 2.286279 [ 6464/60000]
loss: 2.266211 [12864/60000]
loss: 2.261395 [19264/60000]
loss: 2.243052 [25664/60000]
loss: 2.210274 [32064/60000]
loss: 2.217768 [38464/60000]
loss: 2.183949 [44864/60000]
loss: 2.184356 [51264/60000]
loss: 2.152874 [57664/60000]
Test Error: 
 Accuracy:  48.1%, Avg loss: 2.144997 

Epoch 2
--------------
loss: 2.153458 [   64/60000]
loss: 2.145929 [ 6464/60000]
loss: 2.083087 [12864/60000]
loss: 2.100560 [19264/60000]
loss: 2.043809 [25664/60000]
loss: 1.984696 [32064/60000]
loss: 2.014213 [38464/60000]
loss: 1.930321 [44864/60000]
loss: 1.943913 [51264/60000]
loss: 1.875805 [57664/60000]
Test Error: 
 Accuracy:  51.1%, Avg loss: 1.863792 

Epoch 3
--------------
loss: 1.894551 [   64/60000]
loss: 1.869680 [ 6464/60000]
loss: 1.741551 [12864/60000]
loss: 1.793395 [19264/60000]
loss: 1.671934 [25664/60000]
loss: 1.630484 [32064/60000]
loss: 1.662072 [38464/60000]
loss: 1.555714 [44864/60000]
loss: 1.59822

# Save a model

In [35]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")


Saved PyTorch Model State to model.pth


# Load model

In [36]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>

# Making predictions

In [37]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot"
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
  x = x.to(device)
  pred = model(x)
  predicted, actual = classes[pred[0].argmax(0)], classes[y]
  print(f'Predicted: "{predicted}", Actual: "{actual}"')
#

Predicted: "Ankle boot", Actual: "Ankle boot"
